# Experiment: WOP C++ Speed Benchmark

This notebook runs only the C++ implementation and measures runtime/throughput stability.

Expected workspace layout:
- `external_wop_cpp` (C++ project)
                


In [2]:
from __future__ import annotations

import json
import shutil
import statistics
import subprocess
import time
from pathlib import Path

import pandas as pd
from IPython.display import display


def locate_workspace_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "external_wop_cpp").exists():
            return candidate
    raise FileNotFoundError("Could not find workspace root with external_wop_cpp")


WORKSPACE_ROOT = locate_workspace_root(Path.cwd())
CPP_ROOT = WORKSPACE_ROOT / "external_wop_cpp"

print(f"WORKSPACE_ROOT = {WORKSPACE_ROOT}")
print(f"CPP_ROOT      = {CPP_ROOT}")
                


ModuleNotFoundError: No module named 'pandas'

## Prepare C++ CLI

This cell configures and builds `wop_cli` in `external_wop_cpp/build_bench`.
                


In [ ]:
cmake_exe = shutil.which("cmake")
if cmake_exe is None:
    cmake_candidates = [
        Path("C:/Program Files/CMake/bin/cmake.exe"),
        Path("C:/Program Files (x86)/CMake/bin/cmake.exe"),
    ]
    cmake_exe = next((str(p) for p in cmake_candidates if p.exists()), None)

if cmake_exe is None:
    raise FileNotFoundError(
        "CMake is not available in PATH. Install CMake and restart Jupyter, "
        "or add cmake.exe to PATH."
    )

print(f"Using CMake: {cmake_exe}")

subprocess.run([
    cmake_exe,
    "-S",
    ".",
    "-B",
    "build_bench",
    "-DCMAKE_BUILD_TYPE=Release",
    "-DWOP_BUILD_TESTS=OFF",
], cwd=CPP_ROOT, check=True)

subprocess.run([
    cmake_exe,
    "--build",
    "build_bench",
    "--config",
    "Release",
], cwd=CPP_ROOT, check=True)

cpp_candidates = [
    CPP_ROOT / "build_bench" / "Release" / "wop_cli.exe",
    CPP_ROOT / "build_bench" / "Debug" / "wop_cli.exe",
    CPP_ROOT / "build_bench" / "wop_cli.exe",
    CPP_ROOT / "build_bench" / "wop_cli",
]

CPP_CLI = next((p for p in cpp_candidates if p.exists()), None)
if CPP_CLI is None:
    raise FileNotFoundError("wop_cli was not found in external_wop_cpp/build_bench")

CPP_CLI
                


Using CMake: /usr/bin/cmake
-- Configuring done (0.0s)
-- Generating done (0.0s)
-- Build files have been written to: /home/admin/Random_walk_on_plates/external_wop_cpp/build_bench
[  8%] Building CXX object CMakeFiles/wop_core.dir/src/math/vec3.cpp.o
[ 16%] Building CXX object CMakeFiles/wop_core.dir/src/geometry/polyhedron.cpp.o
[ 25%] Building CXX object CMakeFiles/wop_core.dir/src/geometry/box.cpp.o
[ 33%] Building CXX object CMakeFiles/wop_core.dir/src/rng/rng.cpp.o
[ 41%] Building CXX object CMakeFiles/wop_core.dir/src/sampling/sampling.cpp.o
[ 50%] Building CXX object CMakeFiles/wop_core.dir/src/estimation/estimation.cpp.o
[ 58%] Building CXX object CMakeFiles/wop_core.dir/src/solver/wop_solver_internal.cpp.o
[ 66%] Building CXX object CMakeFiles/wop_core.dir/src/solver/wop_solver.cpp.o
[ 75%] Building CXX object CMakeFiles/wop_core.dir/src/solver/wos_box_solver.cpp.o
[ 83%] Linking CXX static library libwop_core.a
[ 83%] Built target wop_core
[ 91%] Building CXX object CMakeFil

PosixPath('/home/admin/Random_walk_on_plates/external_wop_cpp/build_bench/wop_cli')

## Benchmark Helpers

Each C++ run executes `wop_cli --json`, then notebook aggregates timing statistics.
                


In [ ]:
def run_cpp_once(
    n_paths: int,
    seed: int,
    x0: tuple[float, float, float] = (3.0, 0.0, 0.0),
    max_steps: int = 200_000,
    r_max: float = 1e6,
) -> dict[str, float | int]:
    cmd = [
        str(CPP_CLI),
        "--example",
        "box",
        "--x0",
        f"{x0[0]} {x0[1]} {x0[2]}",
        "--n",
        str(int(n_paths)),
        "--seed",
        str(int(seed)),
        "--max-steps",
        str(int(max_steps)),
        "--r-max",
        str(float(r_max)),
        "--json",
    ]
    t0 = time.perf_counter()
    out = subprocess.check_output(cmd, cwd=CPP_ROOT, text=True)
    elapsed = time.perf_counter() - t0
    parsed = json.loads(out)
    parsed["runtime_sec"] = elapsed
    return parsed


def run_case(n_paths: int, seed: int, repeats: int = 5, warmup_runs: int = 1) -> dict[str, float | int]:
    for _ in range(warmup_runs):
        _ = run_cpp_once(n_paths=n_paths, seed=seed)

    samples: list[float] = []
    final_payload: dict[str, float | int] | None = None

    for _ in range(repeats):
        payload = run_cpp_once(n_paths=n_paths, seed=seed)
        samples.append(float(payload["runtime_sec"]))
        final_payload = payload

    assert final_payload is not None

    mean_t = statistics.mean(samples)
    return {
        "n_paths": int(n_paths),
        "runtime_mean_sec": float(mean_t),
        "throughput_paths_per_sec": float(n_paths) / float(mean_t),
        "J": float(final_payload["J"]),
        "eps": float(final_payload["eps"]),
        "n_total": int(final_payload["n_total"]),
        "n_truncated": int(final_payload["n_truncated"]),
    }
                


## Run Benchmark Tests

Adjust `N_VALUES`, `SEEDS`, and `REPEATS` for your machine.
For very large `N` (for example `1_000_000` and above), use fewer `SEEDS` or smaller `REPEATS`.
                


In [ ]:
N_VALUES = [100, 1000, 10000, 100000, 1000000, 10000000]
SEEDS = [314159]
REPEATS = 5
WARMUP_RUNS = 1

rows: list[dict[str, float | int]] = []

for n_paths in N_VALUES:
    for seed in SEEDS:
        row = run_case(
            n_paths=n_paths,
            seed=seed,
            repeats=REPEATS,
            warmup_runs=WARMUP_RUNS,
        )
        rows.append(row)

rows[:3]
                


[{'n_paths': 100,
  'seed': 314159,
  'repeats': 5,
  'runtime_mean_sec': 0.002054015399880882,
  'runtime_median_sec': 0.0021519959991564974,
  'runtime_min_sec': 0.001692273000116984,
  'runtime_max_sec': 0.002272331000312988,
  'throughput_paths_per_sec': 48685.12670635248,
  'J': 0.38510747307613413,
  'eps': 0.12031246075437563,
  'n_total': 100,
  'n_truncated': 50},
 {'n_paths': 1000,
  'seed': 314159,
  'repeats': 5,
  'runtime_mean_sec': 0.00809077919984702,
  'runtime_median_sec': 0.007963944000039191,
  'runtime_min_sec': 0.007932964999781689,
  'runtime_max_sec': 0.008580339000218373,
  'throughput_paths_per_sec': 123597.48984608403,
  'J': 0.35807198096706516,
  'eps': 0.03977940264247798,
  'n_total': 1000,
  'n_truncated': 559},
 {'n_paths': 10000,
  'seed': 314159,
  'repeats': 5,
  'runtime_mean_sec': 0.07741995120013598,
  'runtime_median_sec': 0.07728490000044985,
  'runtime_min_sec': 0.07617660200048704,
  'runtime_max_sec': 0.07847385900004156,
  'throughput_paths_

In [ ]:

df = pd.DataFrame(rows)
df


,n_paths,seed,repeats,runtime_mean_sec,runtime_median_sec,runtime_min_sec,runtime_max_sec,throughput_paths_per_sec,J,eps,n_total,n_truncated
0,100,314159,5,0.002054,0.002152,0.001692,0.002272,48685.126706,0.385107,0.120312,100,50
1,1000,314159,5,0.008091,0.007964,0.007933,0.008580,123597.489846,0.358072,0.039779,1000,559
2,10000,314159,5,0.077420,0.077285,0.076177,0.078474,129165.671703,0.349303,0.012535,10000,5684
3,100000,314159,5,0.531262,0.524583,0.512492,0.567302,188231.096378,0.355754,0.003986,100000,56227
4,1000000,314159,5,5.120182,5.106074,5.100481,5.152770,195305.570627,0.355224,0.001261,1000000,563128
5,10000000,314159,5,51.066168,51.047288,51.030242,51.114072,195824.364680,0.354955,0.000399,10000000,5636156
